In [ ]:
# GitHub token for private clone
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("AIC2026-PACs-token")


In [ ]:
# Fresh repo checkout in /kaggle/working
!rm -rf /kaggle/working/AIC2026-PACs
!git clone https://{secret_value_0}@github.com/AkiyaNguyen/AIC2026-PACs.git


In [ ]:
# Base deps + faster-whisper. Kaggle images usually already have ffmpeg.
!pip install -q -r /kaggle/working/AIC2026-PACs/requirements.txt
!pip install -q -r /kaggle/working/AIC2026-PACs/preprocessing_tools/requirements-asr.txt
!ffmpeg -version | head -n 1


In [ ]:
# Edit dataset roots. Keys = Lxx folder under features/audio and features/asr.
# Same-batch shards (L26_a..e) share one audio-dir / out-dir.
BATCHES = {
    # "L21": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l21-a/video"],
    # "L22": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l22-a/video"],
    # "L23": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l23-a/video"],
    # "L24": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l24-a/video"],
    # "L25": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l25-a/video"],
    "L26": [
        "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_a",
        "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_b",
        "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_c",
        "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_d",
        "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_e"
        ],
    "L27": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l27-a/video"],
    "L28": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l28-a/video"],
    # "L29": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l29/video"],
    # "L30": ["/kaggle/input/datasets/akiyanguyen/pacs-data-l30-a/video"],
}


In [ ]:
# Paths + GPU. WAV is bulky — keep on disk, zip JSONL only later.
import sys
from pathlib import Path

REPO = "/kaggle/working/AIC2026-PACs"
AUDIO_ROOT = "/kaggle/working/features/audio"
ASR_ROOT = "/kaggle/working/features/asr"
DEVICE = "gpu"

missing = []
for batch, dirs in BATCHES.items():
    for d in dirs:
        p = Path(d)
        if not p.is_dir():
            missing.append(d)
            continue
        n = len(list(p.glob("*.mp4"))) or len(list(p.rglob("*.mp4")))
        print(f"{batch}  {d}  mp4_count={n}")
if missing:
    raise SystemExit(f"Not a directory: {missing}")

print("AUDIO_ROOT =", AUDIO_ROOT)
print("ASR_ROOT =", ASR_ROOT)
print("DEVICE =", DEVICE)


In [ ]:
# One CLI call per Lxx (all shards in one argv so Whisper loads once).
import subprocess

for batch, dirs in BATCHES.items():
    audio_dir = f"{AUDIO_ROOT}/{batch}"
    out_dir = f"{ASR_ROOT}/{batch}"
    print(f"========== {batch} → {out_dir} ==========", flush=True)
    cmd = [
        sys.executable, "-m", "tools.extract_transcript",
        *dirs,
        "--audio-dir", audio_dir,
        "--out-dir", out_dir,
        "--device", DEVICE,
    ]
    subprocess.run(cmd, cwd=REPO, check=True)

jsonls = sorted(Path(ASR_ROOT).rglob("*.jsonl"))
print(f"Done: {len(jsonls)} jsonl under {ASR_ROOT}")


In [ ]:
# Zip JSONL only (asr/Lxx/VIDEO_ID.jsonl). Unpack into repo features/asr/.
import zipfile
from IPython.display import FileLink, display

asr_root = Path(ASR_ROOT)
files = [p for p in asr_root.rglob("*.jsonl") if p.is_file()]
out = Path("/kaggle/working/asr.zip")
out.unlink(missing_ok=True)
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in files:
        zf.write(f, arcname=str(f.relative_to(asr_root)))
print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({len(files)} jsonl)")
display(FileLink(str(out)))
